In [11]:
from pathlib import Path
import re

import pandas as pd

## 1. Khai bao duong dan


In [ ]:
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASETS_DIR = PROJECT_ROOT / "datasets"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = PROCESSED_DIR / "student_voice_merged.csv"
OUTPUT_REPORT = REPORT_DIR / "data_merge_report.md"

print("Project root:", PROJECT_ROOT)
print("Output CSV:", OUTPUT_CSV)
print("Output report:", OUTPUT_REPORT)

Project root: d:\AI thuc chien\Student Voice Intelligence
Output CSV: d:\AI thuc chien\Student Voice Intelligence\data\processed\student_voice_merged.csv
Output report: d:\AI thuc chien\Student Voice Intelligence\outputs\reports\data_merge_report.md


## 2. Mapping nhan goc sang nhan doc duoc


In [ ]:
# 0 = neutral, 1 = positive, 2 = negative, 3 = toxic
NEU_SENTIMENT_STD_CODE_MAP = {
    0: 0,
    1: 1,
    2: 2,
    3: 3,
}

UIT_SENTIMENT_STD_CODE_MAP = {
    0: 2,  # UIT raw 0 = negative -> std code 2
    1: 0,  # UIT raw 1 = neutral  -> std code 0
    2: 1,  # UIT raw 2 = positive -> std code 1
}

SENTIMENT_CODE_TO_3CLASS_MAP = {
    0: "neutral",
    1: "positive",
    2: "negative",
    3: "negative",  
}

SENTIMENT_CODE_TO_4CLASS_MAP = {
    0: "neutral",
    1: "positive",
    2: "negative",
    3: "toxic",
}

NEU_TOPIC_MAP = {
    0: "spam",
    1: "news",
    2: "academic",
    3: "others",
    4: "student_services",
    5: "jobs_recruitment",
    6: "personal_affairs",
    7: "social_affairs",
    8: "help_share",
    9: "clubs_events",
}

UIT_TOPIC_MAP = {
    0: "lecturer",
    1: "training_program",
    2: "facilities",
    3: "others",
}

FINAL_COLUMNS = [
    "text",
    "source_dataset",
    "split_original",
    "sentiment_raw",
    "sentiment_std_code",
    "sentiment_std_3class",
    "sentiment_std_4class",
    "topic_raw",
    "topic_std",
    "is_toxic",
    "urgency_level",
]

## 3. Ham tien ich


In [ ]:
def clean_text(value):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def validate_columns(df, required_columns, dataset_name):
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise ValueError(f"{dataset_name} thieu cot: {missing}. Cot hien co: {df.columns.tolist()}")


def map_required(series, mapping, label_name):
    numeric_series = series.astype("Int64")
    mapped = numeric_series.map(mapping)
    unmapped_values = sorted(numeric_series[mapped.isna()].dropna().unique().tolist())
    if unmapped_values:
        raise ValueError(f"{label_name} co nhan chua map: {unmapped_values}")
    return mapped


def summarize_counts(series):
    counts = series.value_counts(dropna=False)
    clean_counts = {}
    for key, value in counts.items():
        clean_key = key.item() if hasattr(key, "item") else key
        clean_counts[clean_key] = int(value)
    return clean_counts

## 4. Doc 6 file CSV goc


In [15]:
INPUT_FILES = [
    {
        "source_dataset": "NEU_ESC",
        "split_original": "train",
        "path": DATASETS_DIR / "NEU-ESC" / "train_set.csv",
    },
    {
        "source_dataset": "NEU_ESC",
        "split_original": "validation",
        "path": DATASETS_DIR / "NEU-ESC" / "val_set.csv",
    },
    {
        "source_dataset": "NEU_ESC",
        "split_original": "test",
        "path": DATASETS_DIR / "NEU-ESC" / "test_set.csv",
    },
    {
        "source_dataset": "UIT_VSFC",
        "split_original": "train",
        "path": DATASETS_DIR / "UIT-VSFC" / "uit_vsfc_train.csv",
    },
    {
        "source_dataset": "UIT_VSFC",
        "split_original": "validation",
        "path": DATASETS_DIR / "UIT-VSFC" / "uit_vsfc_validation.csv",
    },
    {
        "source_dataset": "UIT_VSFC",
        "split_original": "test",
        "path": DATASETS_DIR / "UIT-VSFC" / "uit_vsfc_test.csv",
    },
]

raw_frames = []

for item in INPUT_FILES:
    if not item["path"].exists():
        raise FileNotFoundError(f"Khong tim thay file: {item['path']}")

    df = pd.read_csv(item["path"])
    raw_frames.append({**item, "df": df})
    print(f"{item['source_dataset']} | {item['split_original']}: {df.shape} | {item['path'].name}")

NEU_ESC | train: (23048, 3) | train_set.csv
NEU_ESC | validation: (3305, 3) | val_set.csv
NEU_ESC | test: (6613, 3) | test_set.csv
UIT_VSFC | train: (11426, 3) | uit_vsfc_train.csv
UIT_VSFC | validation: (1583, 3) | uit_vsfc_validation.csv
UIT_VSFC | test: (3166, 3) | uit_vsfc_test.csv


## 5. Chuan hoa tung dataset ve schema chung


In [ ]:
def normalize_neu(df, split_original):
    validate_columns(df, ["text", "sentiment", "classification"], "NEU-ESC")

    out = pd.DataFrame()
    out["text"] = df["text"].apply(clean_text)
    out["source_dataset"] = "NEU_ESC"
    out["split_original"] = split_original

    out["sentiment_raw"] = df["sentiment"].astype("Int64")
    out["topic_raw"] = df["classification"].astype("Int64")

    out["sentiment_std_code"] = map_required(df["sentiment"], NEU_SENTIMENT_STD_CODE_MAP, "NEU sentiment std code").astype("Int64")

    out["sentiment_std_3class"] = map_required(out["sentiment_std_code"], SENTIMENT_CODE_TO_3CLASS_MAP, "NEU sentiment 3-class")
    out["sentiment_std_4class"] = map_required(out["sentiment_std_code"], SENTIMENT_CODE_TO_4CLASS_MAP, "NEU sentiment 4-class")
    out["topic_std"] = map_required(df["classification"], NEU_TOPIC_MAP, "NEU topic")

    out["is_toxic"] = (out["sentiment_std_code"] == 3).astype(int)

    out["urgency_level"] = "low"

    return out[FINAL_COLUMNS]


def normalize_uit(df, split_original):
    validate_columns(df, ["sentence", "sentiment", "topic"], "UIT-VSFC")

    out = pd.DataFrame()
    out["text"] = df["sentence"].apply(clean_text)
    out["source_dataset"] = "UIT_VSFC"
    out["split_original"] = split_original

    out["sentiment_raw"] = df["sentiment"].astype("Int64")
    out["topic_raw"] = df["topic"].astype("Int64")

    out["sentiment_std_code"] = map_required(df["sentiment"], UIT_SENTIMENT_STD_CODE_MAP, "UIT sentiment std code").astype("Int64")

    out["sentiment_std_3class"] = map_required(out["sentiment_std_code"], SENTIMENT_CODE_TO_3CLASS_MAP, "UIT sentiment 3-class")
    out["sentiment_std_4class"] = map_required(out["sentiment_std_code"], SENTIMENT_CODE_TO_4CLASS_MAP, "UIT sentiment 4-class")
    out["topic_std"] = map_required(df["topic"], UIT_TOPIC_MAP, "UIT topic")

    out["is_toxic"] = 0
    out["urgency_level"] = "low"

    return out[FINAL_COLUMNS]

## 6. Normalize va concat

Sau khi moi file da ve cung schema, ta moi concat. Day la diem khac voi viec concat som: concat som de gay nham cot va nham mapping.

In [ ]:
normalized_frames = []

for item in raw_frames:
    if item["source_dataset"] == "NEU_ESC":
        normalized = normalize_neu(item["df"], item["split_original"])
    elif item["source_dataset"] == "UIT_VSFC":
        normalized = normalize_uit(item["df"], item["split_original"])
    else:
        raise ValueError(f"Unknown source_dataset: {item['source_dataset']}")

    normalized_frames.append(normalized)
    print(f"Normalized {item['source_dataset']} | {item['split_original']}: {normalized.shape}")

merged = pd.concat(normalized_frames, ignore_index=True)

rows_before_empty_filter = len(merged)
merged = merged[merged["text"].str.len() > 0].copy()
empty_text_removed = rows_before_empty_filter - len(merged)

print("Merged shape:", merged.shape)
print("Removed empty text rows:", empty_text_removed)
merged.head()

Normalized NEU_ESC | train: (23048, 11)
Normalized NEU_ESC | validation: (3305, 11)
Normalized NEU_ESC | test: (6613, 11)
Normalized UIT_VSFC | train: (11426, 11)
Normalized UIT_VSFC | validation: (1583, 11)
Normalized UIT_VSFC | test: (3166, 11)
Merged shape: (49141, 11)
Removed empty text rows: 0


,text,source_dataset,split_original,sentiment_raw,sentiment_std_code,sentiment_std_3class,sentiment_std_4class,topic_raw,topic_std,is_toxic,urgency_level
0,em xin chào các anh chị em em là sinh viên vừa...,NEU_ESC,train,0,0,neutral,neutral,2,academic,0,low
1,xây dựng mô hình sư phạm chuẩn mực sáng tạo ti...,NEU_ESC,train,0,0,neutral,neutral,2,academic,0,low
2,sao lại ghét cái kiểu làm sai xong khóc lóc ăn...,NEU_ESC,train,2,2,negative,negative,3,others,0,low
3,chào thầy đăng ký học ghép . môn học hiện hệ t...,NEU_ESC,train,0,0,neutral,neutral,2,academic,0,low
4,con bé vẫn hoang mang lắm .,NEU_ESC,train,2,2,negative,negative,3,others,0,low


## 7. Kiem tra chat luong sau gop

Cell nay kiem tra cac loi co the lam hong model:

- Text rong
- Cot bat buoc bi null
- Nhan chua map
- Duplicate text

Duplicate chi duoc bao cao, chua drop tu dong.

In [ ]:
missing_by_column = merged.isna().sum()
duplicate_text_count = int(merged.duplicated(subset=["text"]).sum())

quality_summary = {
    "total_rows": int(len(merged)),
    "empty_text_removed": int(empty_text_removed),
    "duplicate_text_count": duplicate_text_count,
    "rows_by_source": summarize_counts(merged["source_dataset"]),
    "rows_by_split": summarize_counts(merged["split_original"]),
    "sentiment_std_code_distribution": summarize_counts(merged["sentiment_std_code"]),
    "sentiment_3class_distribution": summarize_counts(merged["sentiment_std_3class"]),
    "sentiment_4class_distribution": summarize_counts(merged["sentiment_std_4class"]),
    "topic_distribution": summarize_counts(merged["topic_std"]),
    "toxic_distribution": summarize_counts(merged["is_toxic"]),
    "urgency_distribution": summarize_counts(merged["urgency_level"]),
}

print("Missing by column:")
print(missing_by_column)
print("\nQuality summary:")
for key, value in quality_summary.items():
    print(f"{key}: {value}")

required_nulls = missing_by_column[FINAL_COLUMNS]
if required_nulls.sum() > 0:
    raise ValueError(f"Merged data con null o cot bat buoc:\n{required_nulls[required_nulls > 0]}")

Missing by column:
text                    0
source_dataset          0
split_original          0
sentiment_raw           0
sentiment_std_code      0
sentiment_std_3class    0
sentiment_std_4class    0
topic_raw               0
topic_std               0
is_toxic                0
urgency_level           0
dtype: int64

Quality summary:
total_rows: 49141
empty_text_removed: 0
duplicate_text_count: 1
rows_by_source: {'NEU_ESC': 32966, 'UIT_VSFC': 16175}
rows_by_split: {'train': 34474, 'test': 9779, 'validation': 4888}
sentiment_std_code_distribution: {0: 23471, 2: 12639, 1: 12186, 3: 845}
sentiment_3class_distribution: {'neutral': 23471, 'negative': 13484, 'positive': 12186}
sentiment_4class_distribution: {'neutral': 23471, 'negative': 12639, 'positive': 12186, 'toxic': 845}
topic_distribution: {'others': 15218, 'lecturer': 11607, 'academic': 10512, 'training_program': 3040, 'student_services': 2358, 'personal_affairs': 1478, 'news': 902, 'jobs_recruitment': 808, 'social_affairs': 769, 'fa

## 8. Xem nhanh duplicate

Duplicate khong nhat thiet la loi. Nhung neu cung mot text xuat hien o nhieu split, no co the gay leakage khi train/evaluate. Cell nay giup ban xem mau trung truoc khi quyet dinh xu ly.

In [19]:
duplicate_examples = (
    merged[merged.duplicated(subset=["text"], keep=False)]
    .sort_values(["text", "source_dataset", "split_original"])
    .head(20)
)

duplicate_examples

,text,source_dataset,split_original,sentiment_raw,sentiment_std_code,sentiment_std_3class,sentiment_std_4class,topic_raw,topic_std,is_toxic,urgency_level
44259,"thầy dạy hay , tuy nhiên còn nhiều chỗ chưa th...",UIT_VSFC,train,2,1,positive,positive,0,lecturer,0,low
44383,"thầy dạy hay , tuy nhiên còn nhiều chỗ chưa th...",UIT_VSFC,train,0,2,negative,negative,0,lecturer,0,low


## 9. Export file merged va report


In [20]:
merged.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

report_lines = [
    "# Data Merge Report",
    "",
    "## Output",
    "",
    f"- CSV: `{OUTPUT_CSV.relative_to(PROJECT_ROOT)}`",
    f"- Rows: `{len(merged)}`",
    f"- Columns: `{len(merged.columns)}`",
    "",
    "## Input files",
    "",
]

for item in raw_frames:
    report_lines.append(
        f"- `{item['path'].relative_to(PROJECT_ROOT)}`: {item['df'].shape[0]} rows, {item['df'].shape[1]} columns"
    )

report_lines.extend([
    "",
    "## Schema",
    "",
])

for col in FINAL_COLUMNS:
    report_lines.append(f"- `{col}`")

report_lines.extend([
    "",
    "## Quality checks",
    "",
    f"- Empty text rows removed: `{empty_text_removed}`",
    f"- Duplicate text rows reported, not dropped: `{duplicate_text_count}`",
    "",
    "## Distributions",
    "",
    f"- Rows by source: `{quality_summary['rows_by_source']}`",
    f"- Rows by split: `{quality_summary['rows_by_split']}`",
    f"- Sentiment std code: `{quality_summary['sentiment_std_code_distribution']}`",
    f"- Sentiment 3-class: `{quality_summary['sentiment_3class_distribution']}`",
    f"- Sentiment 4-class: `{quality_summary['sentiment_4class_distribution']}`",
    f"- Topic: `{quality_summary['topic_distribution']}`",
    f"- Toxic: `{quality_summary['toxic_distribution']}`",
    f"- Urgency: `{quality_summary['urgency_distribution']}`",
    "",
    "## Mapping notes",
    "",
    "- `sentiment_std_code` dung chung quy uoc: `0=neutral`, `1=positive`, `2=negative`, `3=toxic`.",
    "- NEU raw sentiment da trung quy uoc chuan: `0=neutral`, `1=positive`, `2=negative`, `3=toxic`.",
    "- UIT raw sentiment duoc doi code truoc khi gop: `0=negative -> 2`, `1=neutral -> 0`, `2=positive -> 1`.",
    "- NEU `sentiment_std_code=3` duoc map thanh `negative` trong `sentiment_std_3class`, thanh `toxic` trong `sentiment_std_4class`, va `is_toxic=1`.",
    "- UIT khong co nhan toxic rieng, nen `is_toxic=0` cho UIT.",
    "- `urgency_level` tam thoi la `low` cho toan bo du lieu. Phase sau co the thay bang rule-based labeling.",
    "- `topic_std` hien la nhan text da chuan hoa tu nhan so cua tung dataset; chua ep NEU va UIT ve mot taxonomy hep de tranh mapping sai.",
])

OUTPUT_REPORT.write_text("\n".join(report_lines), encoding="utf-8")

print(f"Saved CSV: {OUTPUT_CSV}")
print(f"Saved report: {OUTPUT_REPORT}")

Saved CSV: d:\AI thuc chien\Student Voice Intelligence\data\processed\student_voice_merged.csv
Saved report: d:\AI thuc chien\Student Voice Intelligence\outputs\reports\data_merge_report.md
